# Part F · Generalization tests on the Part E models
Uses what is already saved on Google Drive (`sarcasm/models`, `sarcasm/results`, `sarcasm/predictions`), so nothing from Part E is retrained except the two retraining tests (F2, F3, about 3 minutes each).

**How to use:** set `MODEL` in section 2, run all cells. Then change `MODEL` to the other model and run again (save a copy of the notebook in between). The last cell compares the two best models overall and does not depend on `MODEL`.

| Item | Test |
|---|---|
| F1 | prior shift, F1 against threshold |
| F2 | unseen dialect (egypt) |
| F3 | URL and mention shortcuts |
| F4 | removing emojis, laughter, repeated punctuation |
| F5 | bootstrap intervals and McNemar |
| F6 | error analysis |

The test set is used for analysis only, never to choose a model or a threshold.

## 1. Setup

In [ ]:
import os
import sys
from google.colab import drive
drive.mount("/content/drive")

REPO = "Arabic-sarcasm"

if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/cad0m/{REPO}.git /content/{REPO}
else:
    !git -C /content/{REPO} pull

%cd /content/{REPO}

if f"/content/{REPO}" not in sys.path:
    sys.path.insert(0, f"/content/{REPO}")

!pip install -q transformers arabert emoji tashaphyne nltk

Link the three output folders to Drive, as in the training notebooks.

In [ ]:
import shutil
from pathlib import Path

ROOT = Path("/content/Arabic-sarcasm")
DRIVE = Path("/content/drive/MyDrive/sarcasm")

for name in ["results", "predictions", "models"]:
    local, remote = ROOT / name, DRIVE / name
    if not remote.exists():
        remote.mkdir(parents=True)
    if local.exists() and not local.is_symlink():
        shutil.rmtree(local)
    if not local.exists():
        local.symlink_to(remote)

## 2. Data, saved results and the models to test

In [ ]:
import sys
sys.path.insert(0, "/content/Arabic-sarcasm")

import re
import warnings

import emoji
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from transformers import AutoTokenizer
from transformers import logging as hf_logging

from src.data import get_data
from src.evaluate import PREDICTIONS, RESULTS, bootstrap_ci, f1_by_threshold, mcnemar, metrics
from src.fine_tune import DEVICE, PLMS, MultiTask, SingleTask, predict_proba, prepare_text, train_model
from src.train import best_threshold, set_seed

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
warnings.filterwarnings("ignore", message="Detected call of")

MODEL = "arabert"      # "arabert" or "marbert"
PLM = PLMS[MODEL]
MAX_LEN = PLM["max_len"]
SEEDS = [42, 43, 44]

train, val, test, report = get_data()
y_val = val["sarcasm"].to_numpy()
y_test = test["sarcasm"].to_numpy()
print(len(train), len(val), len(test))

What is on Drive for this model? Expected: 4 checkpoints, 12 prediction files, 12 result rows.

In [ ]:
table = pd.read_csv(RESULTS)
rows = table[(table["preprocessing"] == PLM["prep"]) & table["model"].str.contains("_single_|_multi_")]

print("checkpoints:     ", len(list((ROOT / "models").glob(f"{MODEL}_*.pt"))))
print("prediction files:", len(list((ROOT / "predictions").glob(f"{MODEL}_*.csv"))))
print("result rows:     ", len(rows))

Text of the test set, prepared the way this model was trained, and the tokenizer.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PLM["hf_name"])
x_test = prepare_text(MODEL, test["tweet"])

Validation F1 over the 3 seeds. The models tested are E1 and the E2 with the best mean; `best_name` is the better of the two and is the one F2 and F3 retrain.

In [ ]:
results = rows
val_table = results.groupby("model")["val_f1"].agg(["mean", "std", "count"])
display(val_table)

best_e2 = val_table[val_table.index.str.contains("_multi_")]["mean"].idxmax()
chosen = [f"{MODEL}_single_lam0", best_e2]
best_name = val_table.loc[chosen, "mean"].idxmax()

task = "single" if "_single_" in best_name else "multi"
lam = 0 if task == "single" else float(best_name.split("lam")[1])
print("tested:", chosen, "| best:", best_name)

## 3. Helpers

In [ ]:
def make_model(task, lam):
    if task == "single":
        model = SingleTask(PLM["hf_name"])
    else:
        model = MultiTask(PLM["hf_name"], lam)
    return model.to(DEVICE)


def load_checkpoint(name, seed=42):
    ckpt = torch.load(ROOT / "models" / f"{name}_seed{seed}.pt", map_location=DEVICE)
    model = make_model(ckpt["task"], ckpt["lam"])
    model.load_state_dict(ckpt["state_dict"])
    return model, ckpt["threshold"]


def load_pred(name, seed=42):
    pred = pd.read_csv(PREDICTIONS / f"{name}_seed{seed}.csv")
    assert (pred["id"].to_numpy() == test["id"].to_numpy()).all(), "predictions are not in the test order"
    return pred


def saved_threshold(name, seed=42):
    row = results[(results["model"] == name) & (results["seed"] == seed)]
    return row["threshold"].iloc[0]


def fit_and_predict(train_df, val_df, x_eval):
    # retrain the best model (same task, lam, seed 42); threshold chosen on val_df; x_eval is already prepared text
    set_seed(42)
    model = make_model(task, lam)
    x_t = prepare_text(MODEL, train_df["tweet"])
    x_v = prepare_text(MODEL, val_df["tweet"])
    y_v = val_df["sarcasm"].to_numpy()

    train_model(model, tokenizer, x_t, train_df["sarcasm"].to_numpy(), train_df["sentiment_id"].to_numpy(),
                x_v, y_v, MAX_LEN)
    threshold = best_threshold(y_v, predict_proba(model, tokenizer, x_v, MAX_LEN))
    proba = predict_proba(model, tokenizer, x_eval, MAX_LEN)

    del model
    torch.cuda.empty_cache()
    return threshold, proba

## F1 · prior shift
Sarcasm rate in each split, then validation F1 against test F1 (the test uses the threshold chosen on validation).

In [ ]:
print(f"sarcasm rate: train {train['sarcasm'].mean():.1%} | val {y_val.mean():.1%} | test {y_test.mean():.1%}")
results[results["model"].isin(chosen)].groupby("model")[["val_f1", "test_f1"]].mean()

F1 against the threshold, on the test. **Analysis only**: the dotted line is the threshold chosen on validation, the peak is what the test would have preferred.

In [ ]:
for name in chosen:
    pred = load_pred(name)
    curve = f1_by_threshold(pred["y_true"], pred["proba"])
    val_thr = saved_threshold(name)

    line, = plt.plot(curve.index, curve.values, label=name)
    plt.axvline(val_thr, color=line.get_color(), linestyle=":")

    f1_at_val = f1_score(pred["y_true"], pred["y_pred"])
    print(f"{name}: val threshold {val_thr:.2f} -> test F1 {f1_at_val:.4f} | best test threshold {curve.idxmax():.2f} -> {curve.max():.4f}")

plt.xlabel("threshold")
plt.ylabel("test F1 (sarcastic class)")
plt.legend()
plt.show()

## F2 · unseen dialect
Retrain the best model without `egypt` (also removed from validation, so the threshold never sees it) and compare on the egypt tweets of the test. About 3 minutes.

In [ ]:
train_no_egypt = train[train["dialect"] != "egypt"]
val_no_egypt = val[val["dialect"] != "egypt"]
print("train:", len(train), "->", len(train_no_egypt))

thr_no, proba_no = fit_and_predict(train_no_egypt, val_no_egypt, x_test)

egypt = (test["dialect"] == "egypt").to_numpy()
full = load_pred(best_name)
f1_full = metrics(y_test[egypt], full["proba"].to_numpy()[egypt], saved_threshold(best_name))["f1"]
f1_no = metrics(y_test[egypt], proba_no[egypt], thr_no)["f1"]

print("egypt tweets in test:", egypt.sum())
print(f"F1 on egypt, trained on all train:  {f1_full:.4f}")
print(f"F1 on egypt, trained without egypt: {f1_no:.4f}   (change {f1_no - f1_full:+.4f})")

## F3 · shortcuts
URL and mention patterns. **Check the counts printed below**: if they are 0 or look wrong, adjust the two patterns after looking at the raw tweets.

In [ ]:
URL = r"https?://\S+|www\.\S+|\bURL\b"
MENTION = r"@\w+|\bUSER\b"

has_url = test["tweet"].str.contains(URL).to_numpy()
has_mention = test["tweet"].str.contains(MENTION).to_numpy()
print("test tweets with URL:", has_url.sum(), "| with mention:", has_mention.sum())
print(f"sarcasm rate with URL {y_test[has_url].mean():.1%} | without URL {y_test[~has_url].mean():.1%}")

for name in chosen:
    pred = load_pred(name)
    y, y_pred = pred["y_true"].to_numpy(), pred["y_pred"].to_numpy()
    print(f"{name}: F1 with URL {f1_score(y[has_url], y_pred[has_url]):.4f} | without URL {f1_score(y[~has_url], y_pred[~has_url]):.4f}")

Retrain on text without URL and mention tokens (removed before the model-specific preparation), and evaluate on the test text prepared the same way. About 3 minutes.

In [ ]:
def strip_tokens(tweets):
    return tweets.str.replace(URL + "|" + MENTION, " ", regex=True)


train_s = train.assign(tweet=strip_tokens(train["tweet"]))
val_s = val.assign(tweet=strip_tokens(val["tweet"]))
x_test_s = prepare_text(MODEL, strip_tokens(test["tweet"]))

thr_s, proba_s = fit_and_predict(train_s, val_s, x_test_s)

predictions = {"original text": load_pred(best_name)["y_pred"].to_numpy(),
               "URL and mention removed": (proba_s >= thr_s).astype(int)}
for label, y_pred in predictions.items():
    print(f"{label:25} F1 {f1_score(y_test, y_pred):.4f} | with URL {f1_score(y_test[has_url], y_pred[has_url]):.4f} | without URL {f1_score(y_test[~has_url], y_pred[~has_url]):.4f}")

## F4 · sarcasm cues
Reload the saved checkpoints (seed 42), remove one cue at a time from the sarcastic tweets they detected correctly, and see how many are still detected. The first row removes nothing: it must be close to 100%, otherwise the checkpoint did not load properly.

In [ ]:
CUES = {
    "emoji": lambda t: emoji.replace_emoji(t, ""),
    "laughter": lambda t: re.sub(r"ه{3,}|(?:ها){2,}|(?:هه){2,}", " ", t),
    "repeated punctuation": lambda t: re.sub(r"[!?؟.…]{2,}", " ", t),
}
CUES["all three"] = lambda t: CUES["repeated punctuation"](CUES["laughter"](CUES["emoji"](t)))
steps = [("no change (check)", lambda t: t)] + list(CUES.items())

rows = []
for name in chosen:
    model, thr = load_checkpoint(name)
    pred = load_pred(name)
    found = ((pred["y_true"] == 1) & (pred["y_pred"] == 1)).to_numpy()
    found_tweets = test["tweet"].to_numpy()[found]

    for cue, remove in steps:
        if cue == "no change (check)":
            subset = list(found_tweets)
        else:
            subset = [t for t in found_tweets if remove(t) != t]       # only tweets that contain the cue
        x = prepare_text(MODEL, [remove(t) for t in subset])
        still = (predict_proba(model, tokenizer, x, MAX_LEN) >= thr).mean()
        rows.append({"model": name, "cue removed": cue, "tweets": len(subset), "still detected": round(still, 3)})

    del model
    torch.cuda.empty_cache()

pd.DataFrame(rows)

## F5 · significance
F1 with a 95% bootstrap interval (1,000 resamples) for each of the 3 seeds, then the mean and std over seeds.

In [ ]:
rows = []
for name in chosen:
    for seed in SEEDS:
        pred = load_pred(name, seed)
        low, high = bootstrap_ci(pred["y_true"], pred["y_pred"])
        rows.append({"model": name, "seed": seed, "f1": f1_score(pred["y_true"], pred["y_pred"]),
                     "ci_low": low, "ci_high": high})

ci = pd.DataFrame(rows)
print(ci.round(4).to_string(index=False))
print()
print(ci.groupby("model")["f1"].agg(["mean", "std"]).round(4))

McNemar test between E1 and the chosen E2, seed by seed.

In [ ]:
a, b = chosen

for seed in SEEDS:
    pa, pb = load_pred(a, seed), load_pred(b, seed)
    p_value = mcnemar(pa["y_true"], pa["y_pred"], pb["y_pred"])
    print(f"seed {seed}: McNemar p-value, {a} vs {b} = {p_value:.4f}")

## F6 · error analysis
15 false negatives and 15 false positives of the best model, with their dialect and whether the tweet belongs to a group of duplicates with contradictory labels (`conflicts.csv`). The sample is saved so you can fill the `category` column.

In [ ]:
pred = load_pred(best_name)
errors = test.assign(proba=pred["proba"].to_numpy(), y_pred=pred["y_pred"].to_numpy())
errors = errors[errors["sarcasm"] != errors["y_pred"]]

conflicts = pd.read_csv(ROOT / "results" / "conflicts.csv")
errors = errors.assign(in_conflict=errors["key"].isin(conflicts["key"]))

print("errors:", len(errors), "| false negatives:", (errors["sarcasm"] == 1).sum(), "| false positives:", (errors["sarcasm"] == 0).sum())
print("errors inside contradictory-label groups:", errors["in_conflict"].sum())

fn = errors[errors["sarcasm"] == 1].sample(15, random_state=42)
fp = errors[errors["sarcasm"] == 0].sample(15, random_state=42)
sample = pd.concat([fn, fp])[["id", "sarcasm", "proba", "dialect", "in_conflict", "tweet"]].assign(category="")
sample.to_csv(ROOT / "results" / f"errors_{best_name}.csv", index=False)
sample

Fill `category` in the saved CSV with one of: world knowledge or context needed · irony without marker · marker misread · dialect · annotation noise. Then count them:

In [ ]:
labeled = pd.read_csv(ROOT / "results" / f"errors_{best_name}.csv")
labeled["category"].value_counts()

## F5 (end) · the two best models overall
Across both models (this cell does not depend on `MODEL`): the two configurations with the best mean validation F1, compared with McNemar on the test, seed by seed.

In [ ]:
everything = pd.read_csv(RESULTS)
everything = everything[everything["model"].str.contains("_single_|_multi_")]
ranking = everything.groupby("model")["val_f1"].mean().sort_values(ascending=False)
print(ranking.round(4))

a, b = ranking.index[:2]
for seed in SEEDS:
    pa, pb = load_pred(a, seed), load_pred(b, seed)
    p_value = mcnemar(pa["y_true"], pa["y_pred"], pb["y_pred"])
    print(f"seed {seed}: McNemar p-value, {a} vs {b} = {p_value:.4f}")